# Game Engine

In this notebook, we're going to complete the objective for week three: **building the engine**. The engine will be based on our flagship version of an LLM—Gemini 3 Flash—which was prompted using few-shot, chain of thought (CoT), and versioning. Additionally, we'll be supporting input and output (I/O) from the user. This means:

* Piping the relevant information from our latest few-shot file
* Creating an architecture that incorporates our setlist of skits
* Defining the logic behind the branching (if any)
* Toying with other prompt practices to make NPC interactions better

In [1]:
# Install the official Google Generative AI SDK ('-q' silences the output, '-U' ensures the most up-to-date version)
!pip install -q -U google-generativeai

In [2]:
# Add your dependencies (may need more/less; TBD)
import textwrap # new line
#import random
import re
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Fetch API key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key = GEMINI_API_KEY)

In [3]:
# Interpolate the few-shot-prompted LLM to build the evaluator
evaluator_instruction = (
    "You are a professional comedy writer. Evaluate whether the text is funny or not based on this binary rubric:\n"
    "0 (not funny) if there's no comedic intent, missing a punchline, feels forced, too niche, or unoriginal.\n"
    "1 (funny) if it's unpredictable yet clever, witty and unique, intuitive in its wordplay, or isn't overty offensive.\n"
    "First, write one brief sentence explaining your reasoning. Then, on a new line, output the final score in this exact format: 'Score: [0/1]'."
)

few_shot_examples = [
    # Hard-coded three examples of class 0 & reasons for their classification
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: I used to date Hispanic guys, but now I prefer consensual.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is a racist rape joke mocking Hispanic men, making it both overly crude and insensitive. \nScore: 0")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: You can get a lot of power out of one inch.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is low and almost too easy of a joke to make, so it's unoriginal and not funny. \nScore: 0")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: No one notices the Taylor oil spill because it's a disaster taking place over a long period of time, like Derrick Rose's career.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This comes off as antagonistic since it throws shade at a person, Derrick Rose (whom many consider to be a great but flawed NBA player), and is too unpopular of an opinion, thus not constituting a good joke. \nScore: 0")]),

    # Hard-coded three examples of class 1 & reasons for their classification
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Please, don't call me sir. Call me ma'am.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because it is witty and self-deprecating, given that the user is indeed a man. \nScore: 1")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Hey, I am Conan O'Brien and I'm honored to be the last human host of the Academy Awards. Yes! Yeah! Next year, it's going to be a Waymo in a tux.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is very funny as it's clever and uses observational comedy to comment on the advent of AI in self-driving cars, like Waymo, whilst keeping it wholesome. \nScore: 1")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: How did us teaching Diana to drive turn into I'm a male prostitute, you're going to put me out, and you're going to come back in an hour, and you want your trap full?")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because, in spite of its unorthodox format (as it doesn't read like a classic joke or pun), it calls out the absurdity of the situation in a unique and unpredictable way. \nScore: 1")])
]

def get_humor_score(player_text):
    turn_input = types.Content(role = 'user', parts = [types.Part.from_text(text = f"Text: {player_text}")])
    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = few_shot_examples + [turn_input],
        config = types.GenerateContentConfig(
            system_instruction = evaluator_instruction,
            temperature = 0
        )
    )
    match = re.search(r'Score:\s*([0-1])', response.text)
    return int(match.group(1)) if match else 0

In [4]:
# Create the game / dungeon master whose job is to add to & connect the skits
def skitzo(skit_skeleton):
    prompt = f"""You are the (dungeon) master for an interactive text-based game in the style of 'Choice of Games.' Your job is to take a basic outline of a skit and expand it into an immersive RPG scene using the following pointers:

    1. Narrate in the second person (e.g. 'You pull up to...' or 'Your heart races.')
    2. Write one to four paragraphs totalling no more than a dozen sentences. Let the complexity of the outline dictate the exact length, just like a real novel
    3. Ground the player in the scene by describing the setting (i.e. one or more of the five senses—sight, smell, touch, taste, hearing)
    4. Include at least one piece of dialogue from an NPC interwoven into the scene
    5. End the final paragraph with an NPC interacting directly to the player. Leave the narrative hanging so that the player must respond.

    Here's the outline to expand upon: {skit_skeleton}"""

    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            temperature = 0.7
        )
    )

    return response.text

def npc(scenario, player_text, score):
    reaction_type = "positive, laughing, and rewarding" if score == 1 else "awkward, offended, or confused"
    prompt = f"""
    Context: {scenario}
    Player's response: {player_text}

    The player's response was evaluated for humor and got a score of {score}/1. Write a short, {reaction_type} response from the NPC reacting to the player."""

    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            temperature = 0.7
        )
    )

    return response.text

In [5]:
# Initialize the skits
skits = [
    "The user is a college exchange student who wants to sound like a native, so they take a crash course in a foreign language—with special emphasis on learning slang—from an instructor and dialect coach. It is provided that the LLM gives translations for the other tongue in parentheses (akin to subtitles) so that the user may respond in English.",
    "The user crashes a club meeting they have never been to before. Is it for freebies? Is it for…um, kicks and giggles? Maybe getting a guy or girl’s number or something?",
    "The user takes a part-time job at a convenience store, cafe, or cram school to make some extra money on the side. And because they want that Employee of the Month (EOM) reward. For a genuine, authentic experience, it sure is a hell of a lot of work.",
    "The user surprises their older brother who is in the middle of a date with his new girlfriend. Like every younger sibling, they proceed to third-wheel for the rest of the day. Poor child…",
    "The user is about to finish their exchange program, so they try to cook their homestay dinner as a thank you. Time for a solo visit to the grocery store with a recipe they found on YouTube."
]

In [8]:
# Run the game
def run_game():
    print("Insert cold open here...\n")
    total_score = 0
    num_rounds = len(skits) # new line

    #for round in range(1, 6):
    for round in range(num_rounds):
        #print("=" * 50 + f"\nROUND {round}\n" + "=" * 50)
        print("=" * 100) # new line
        print(f"ROUND {round + 1}") # new line
        print("=" * 100 + "\n") # new line

        #skeleton = random.choice(skits)
        skeleton = skits[round]
        full_skit = skitzo(skeleton)
        wrapped_skit = textwrap.fill(full_skit, width = 100) # Will delete later before connecting to Streamlit
        #print(full_skit)
        print(wrapped_skit) # new line

        player_action = input("\nWhat do you do/say? > ")

        score = get_humor_score(player_action)
        total_score += score

        reaction = npc(full_skit, player_action, score)
        wrapped_reaction = textwrap.fill(reaction, width = 100) # Will delete later before connecting to Streamlit
        print(f"\n[Evaluator Score: {score}/1]") # May want to delete this later so it doesn't show
        #print(f"NPC: {reaction}\n")
        print(f"NPC: {wrapped_reaction}\n") # new line

        time.sleep(1)

    # Define the endings from the branching
    print("=" * 50 + "\nGAME OVER\n" + "=" * 50)
    if total_score == 5:
        print("You did it.")
    elif total_score > 2:
        print("You were aight.")
    else:
        print("You suck.")

In [9]:
# Start here!
run_game()

Insert cold open here...

ROUND 1

The afternoon sun bakes the cobblestones outside the narrow window, and the scent of burnt espresso
clings to the stacks of yellowed papers on the desk. You sit across from Professor Moretti, whose
eyes twinkle with a mix of mischief and academic rigor. He leans forward, the leather of his chair
creaking as he taps a rhythmic beat on the wooden tabletop.  "Textbooks are for tourists who want to
find the bathroom," he says, dismissively waving a hand at your pristine grammar guide. "If you want
to survive the streets of this city without sounding like a talking dictionary, you must embrace the
grit of the vernacular." He scribbles a few phrases on a napkin, his pen scratching loudly in the
quiet room.  He slides the napkin toward you and watches your reaction intently. "Imagine you're at
a crowded bar and someone cuts in line; you don't say 'excuse me,' you say *¡No te pases de listo!*
(Don't try to be a smart-ass!) with just the right amount of teeth.